In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score


In [20]:
df = pd.read_csv(
    "/content/rows.csv",
    engine="python",
    on_bad_lines="skip"
)


In [22]:
df.shape



(74525, 18)

In [23]:
df.shape
df.columns


Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')

In [24]:
df.isna().sum()


,0
Date received,0
Product,0
Sub-product,0
Issue,0
Sub-issue,9531
Consumer complaint narrative,62166
Company public response,38984
Company,0
State,2465
ZIP code,10774


In [25]:
df[[TEXT_COL, LABEL_COL]].isna().sum()


,0
Consumer complaint narrative,62166
Product,0


In [27]:
pd.DataFrame({
    "not_null": df[[TEXT_COL, LABEL_COL]].notna().sum(),
    "null": df[[TEXT_COL, LABEL_COL]].isna().sum()
})


,not_null,null
Consumer complaint narrative,12359,62166
Product,74525,0


In [28]:
TEXT_COL  = "Consumer complaint narrative"
LABEL_COL = "Product"

df = df[[TEXT_COL, LABEL_COL]]


In [29]:
df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str)


In [30]:
from sklearn.model_selection import train_test_split

X = df[TEXT_COL]
y = df[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.9,
    min_df=5,
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)


In [35]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(
    loss="hinge",     # 👈 only this line changes
    max_iter=1000,
    tol=1e-3,
    random_state=42
)


model.fit(X_train_tfidf, y_train)


SGDClassifier(random_state=42)

In [36]:
y_pred = model.predict(X_test_tfidf)


In [37]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy : 0.8252427184466019
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.78      0.83      0.81       151
                                                 Credit card or prepaid card       0.87      0.77      0.81       239
Credit reporting, credit repair services, or other personal consumer reports       0.85      0.91      0.88      1026
                                                             Debt collection       0.78      0.82      0.80       565
                          Money transfer, virtual currency, or money service       0.82      0.61      0.70        46
                                                                    Mortgage       0.86      0.90      0.88       205
                                   Payday loan, title loan, or personal loan       0.82      0.16      0.27        55
                         

In [38]:
new_complaint = [
    "I was charged extra interest on my credit card and the bank is not resolving my issue"
]

new_complaint_tfidf = tfidf.transform(new_complaint)

pred = model.predict(new_complaint_tfidf)

print("Predicted Product :", pred[0])


Predicted Product : Credit card or prepaid card
